In [1]:
from google.colab import drive
from pathlib import Path
import torch

%pip install ultralytics --quiet
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics", "--quiet"], check=True)
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### 1. Configuración Básica y verificar que el archivo .yaml y train.txt existen 

In [3]:
# Montar Google Drive
drive.mount("/content/drive")

# Ruta de la carpeta en Google Drive
BASE_DIR = "toy_dataset/"
DATA_YAML = "/content/drive/MyDrive/toy_dataset/data.yaml"          # Archivo .yaml de tu dataset
BASE_MODEL = "yolo11n.pt"                                           # Modelo base ligero para entrenar rápido
TRAIN_NAME = "/content/drive/MyDrive/toy_dataset/train.txt"

PROJECT_NAME = "runs_toy"
RUN_NAME = "toy_yolo_v1"

if not Path(DATA_YAML).exists():
    raise FileNotFoundError(f"No se encontró el archivo {DATA_YAML}")

if not Path(TRAIN_NAME).exists():
    raise FileNotFoundError("No se encontró train.txt")

Mounted at /content/drive


### 3. Cargar modelo base

In [4]:
model = YOLO(BASE_MODEL)

In [9]:
import os

train_txt = "/content/drive/MyDrive/toy_dataset/train.txt"

with open(train_txt, "r") as f:
    lines = f.readlines()

fixed_lines = []
for line in lines:
    line = line.strip()
    if line and not line.startswith("/"):
        # Le añade la / inicial si le falta
        line = "/" + line
    fixed_lines.append(line + "\n")

with open(train_txt, "w") as f:
    f.writelines(fixed_lines)

print(f"✅ train.txt revisado. Primeras 3 líneas:")
for l in fixed_lines[:3]:
    print(" ", l.strip())

✅ train.txt revisado. Primeras 3 líneas:
  /content/drive/MyDrive/toy_dataset/images/train/frame_000000.jpg
  /content/drive/MyDrive/toy_dataset/images/train/frame_000001.jpg
  /content/drive/MyDrive/toy_dataset/images/train/frame_000002.jpg


In [ ]:
from pathlib import Path

train_txt = "/content/drive/MyDrive/toy_dataset/train.txt"

with open(train_txt) as f:
    paths = [l.strip() for l in f if l.strip()] # Elimina líneas vacías 

missing = [p for p in paths if not Path(p).exists()] # Verifica si cada path existe
print(f"Total imágenes en txt: {len(paths)}")
print(f"Imágenes que SÍ existen: {len(paths) - len(missing)}")
print(f"Imágenes faltantes: {len(missing)}")
if missing:
    print("Ejemplo de path que no existe:", missing[0])

Total imágenes en txt: 578
Imágenes que SÍ existen: 578
Imágenes faltantes: 0


In [11]:
import yaml

data_yaml_path = "/content/drive/MyDrive/toy_dataset/data.yaml"

config = {
    "names": {0: "Toy"},
    "path": "/content/drive/MyDrive/toy_dataset",
    "train": "train.txt",
    "val": "train.txt"
}

with open(data_yaml_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("✅ data.yaml actualizado correctamente")

✅ data.yaml actualizado correctamente


### 4. Entrenar modelo

In [12]:
EPOCHS = 80
IMG_SIZE = 640
BATCH_SIZE = 8

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=20,
    project=PROJECT_NAME,
    name=RUN_NAME,
    workers=0
)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/toy_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=toy_yolo_v1-5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_ma

### 5. Subir best.pt a Google Drive

In [ ]:
import shutil

origen = "/content/runs/detect/runs_toy/toy_yolo_v1-5/weights/best.pt"
destino = "/content/drive/MyDrive/toy_dataset/best.pt"

shutil.copy(origen, destino)

print("Modelo copiado a Google Drive:")
print(destino)